In [43]:
import sys
import json
import pandas as pd
from datetime import datetime
import mysql.connector

# -----------------------------
# 1. Receber caminho do ficheiro
# -----------------------------
file_path = sys.argv[1]

# -----------------------------
# 2. Ler Excel com pandas
# -----------------------------
df = pd.read_excel("teste.xls", dtype=str)
df = df.fillna("")  # evitar None

# -----------------------------
# 3. Conectar ao MySQL
# -----------------------------
db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="cirurgia_app"
)

cursor = db.cursor(dictionary=True)

# -----------------------------
# Funções auxiliares
# -----------------------------


def parse_date(value):
    if value is None:
        return None

    # pandas datetime → converter diretamente
    if hasattr(value, "strftime"):
        return value.strftime("%Y-%m-%d")

    value = str(value).strip()
    if value == "":
        return None

    # dd/mm/yyyy
    try:
        if "/" in value:
            return datetime.strptime(value, "%d/%m/%Y").strftime("%Y-%m-%d")
    except:
        pass

    # yyyy-mm-dd hh:mm:ss
    try:
        return datetime.strptime(value, "%Y-%m-%d %H:%M:%S").strftime("%Y-%m-%d")
    except:
        pass

    # yyyy-mm-dd
    try:
        return datetime.strptime(value, "%Y-%m-%d").strftime("%Y-%m-%d")
    except:
        pass

    # Excel serial number
    try:
        return pd.to_datetime(float(value), unit='d', origin='1899-12-30').strftime("%Y-%m-%d")
    except:
        return None


from datetime import datetime, date

def normalize_date(value):
    if value is None:
        return None

    # Já é datetime.date ou datetime.datetime
    if isinstance(value, (datetime, date)):
        return value.strftime("%Y-%m-%d")

    # Converter para string
    value = str(value).strip()
    if value == "":
        return None

    # yyyy-mm-dd hh:mm:ss
    try:
        return datetime.strptime(value, "%Y-%m-%d %H:%M:%S").strftime("%Y-%m-%d")
    except:
        pass

    # yyyy-mm-dd
    try:
        return datetime.strptime(value, "%Y-%m-%d").strftime("%Y-%m-%d")
    except:
        pass

    # dd/mm/yyyy
    try:
        if "/" in value:
            return datetime.strptime(value, "%d/%m/%Y").strftime("%Y-%m-%d")
    except:
        pass

    # Excel serial number
    try:
        return pd.to_datetime(float(value), unit='d', origin='1899-12-30').strftime("%Y-%m-%d")
    except:
        return None





In [44]:

display(df.head(10))

,NUM_LISTA_ESPERA,DTA_MARCACAO,PRIORIDADE,Regime,Situacao,ESTADO,DTA_OPERADO,DTA_AGENDA,NUM_PROCESSO,SEXO,DES_GRUPO,COD_MEDICO,NOME_CLINICO,PATOLOGIA,DES_DIAGNOSTICO,INTERV_CIRURGICA,DTA_CANCEL,CANCEL,DES_CANCEL
0,348080,2025-01-02 00:00:00,1,AMBULATORIO,Operado,F,2025-04-01 00:00:00,,38486,2,HSA - OFTALMOLOGIA,74581,CAROLINA MOTA,H2512,"Catarata senil nuclear, olho ESQ",08DK3ZZ,,,
1,348075,2025-01-02 00:00:00,1,AMBULATORIO,Operado,F,2025-03-21 00:00:00,,2011881,1,HSA - OFTALMOLOGIA,62706,JOAO ABREU CHAVES,H2512,"Catarata senil nuclear, olho ESQ",08RK3JZ,,,
2,348089,2025-01-02 00:00:00,1,AMBULATORIO,Operado,F,2025-03-25 00:00:00,,1573030,1,HSA - OFTALMOLOGIA,45996,SARA SILVA,H2512,"Catarata senil nuclear, olho ESQ",08RK3JZ,,,
3,348094,2025-01-02 00:00:00,1,AMBULATORIO,Operado,F,2025-03-25 00:00:00,,2018239,2,HSA - OFTALMOLOGIA,45996,SARA SILVA,H2511,"Catarata senil nuclear, olho DIR",08RJ3JZ,,,
4,348044,2025-01-02 00:00:00,1,AMBULATORIO,Operado,F,2025-03-21 00:00:00,,202495,2,HSA - OFTALMOLOGIA,28395,RUI MIGUEL CASTRO,H2511,"Catarata senil nuclear, olho DIR",08RJ3JZ,,,
5,348082,2025-01-02 00:00:00,1,AMBULATORIO,Operado,F,2025-03-17 00:00:00,,31400,2,HSA - OFTALMOLOGIA,47499,MONICA MIGUEL SANTOS,H25012,"Catarata senil cortical, olho ESQ",08DK3ZZ,,,
6,348076,2025-01-02 00:00:00,1,AMBULATORIO,Operado,F,2025-04-01 00:00:00,,306854,1,HSA - OFTALMOLOGIA,62706,JOAO ABREU CHAVES,H25041,"Catarata senil subcapsular polar POST, olho DIR",08RJ3JZ,,,
7,348096,2025-01-02 00:00:00,1,AMBULATORIO,Operado,F,2025-03-31 00:00:00,,84258,1,HSA - OFTALMOLOGIA,62706,JOAO ABREU CHAVES,H2511,"Catarata senil nuclear, olho DIR",08RJ3JZ,,,
8,348093,2025-01-02 00:00:00,1,AMBULATORIO,Operado,F,2025-03-25 00:00:00,,2113995,1,HSA - OFTALMOLOGIA,45996,SARA SILVA,H2511,"Catarata senil nuclear, olho DIR",08RJ3JZ,,,
9,348069,2025-01-02 00:00:00,1,AMBULATORIO,Operado,F,2025-03-21 00:00:00,,1196109,1,HSA - OFTALMOLOGIA,62706,JOAO ABREU CHAVES,H2511,"Catarata senil nuclear, olho DIR",08RJ3JZ,,,


In [45]:

def get_existing(id):
    cursor.execute("SELECT * FROM waiting_list WHERE id = %s", (id,))
    return cursor.fetchone()


def insert_new(data):
    fields = ", ".join(data.keys())
    placeholders = ", ".join(["%s"] * len(data))
    values = list(data.values())

    cursor.execute(
        f"INSERT INTO waiting_list ({fields}) VALUES ({placeholders})",
        values
    )
    db.commit()


def update_existing(id, data):
    updates = ", ".join([f"{k} = %s" for k in data.keys()])
    values = list(data.values()) + [id]

    cursor.execute(
        f"UPDATE waiting_list SET {updates} WHERE id = %s",
        values
    )
    db.commit()


def save_history(waiting_list_id, field, old, new):
    cursor.execute("""
        INSERT INTO waiting_list_history
        (waiting_list_id, campo_alterado, valor_antigo, valor_novo, alterado_em, origem)
        VALUES (%s, %s, %s, %s, NOW(), 'excel')
    """, (waiting_list_id, field, old, new))
    db.commit()



In [46]:

# -----------------------------
# 4. Processar Excel
# -----------------------------
imported = 0
updated = 0
unchanged = 0

for _, row in df.iterrows():

    num = str(row["NUM_LISTA_ESPERA"]).strip()

    if not num.isdigit():
        continue

    id = int(num)
    
    if row["DES_GRUPO"] != "HSA - CIRURGIA":
        continue

    data = {
        "id": id,
        "data_marcacao": parse_date(row["DTA_MARCACAO"]),
        "data_operado": parse_date(row["DTA_OPERADO"]),
        "data_agenda": parse_date(row["DTA_AGENDA"]),
        "data_cancel": parse_date(row["DTA_CANCEL"]),
        "prioridade": row["PRIORIDADE"],
        "regime": row["Regime"],
        "situacao": row["Situacao"],
        "estado": row["ESTADO"],
        "num_processo": row["NUM_PROCESSO"],
        "sexo": row["SEXO"],
        "des_grupo": row["DES_GRUPO"],
        "cod_medico": row["COD_MEDICO"],
        "nome_clinico": row["NOME_CLINICO"],
        "patologia": row["PATOLOGIA"],
        "des_diagnostico": row["DES_DIAGNOSTICO"],
        "interv_cirurgica": row["INTERV_CIRURGICA"],
        "cancel": row["CANCEL"],
        "des_cancel": row["DES_CANCEL"],
    }
    
    
    existing = get_existing(id)

    if not existing:
        insert_new(data)
        imported += 1
        continue

    changed = False

    # Comparar todos os campos
    for field, new_value in data.items():
        old_value = existing[field]

        if "data" in field:
            old_norm = normalize_date(old_value)
            new_norm = normalize_date(new_value)
        else:
            old_norm = str(old_value or "")
            new_norm = str(new_value or "")

        if old_norm != new_norm:
            changed = True
            save_history(id, field, old_norm, new_norm)

    # Só atualizar se houve alterações
    if changed:
        # valor novo
        new_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # valor antigo
        old_ts = existing.get("updated_from_excel_at")

        # guardar no histórico também
        save_history(id, "updated_from_excel_at", str(old_ts or ""), new_ts)

        # atualizar BD
        data["updated_from_excel_at"] = new_ts
        update_existing(id, data)

        updated += 1
    else:
        unchanged += 1
    
    continue


# -----------------------------
# 5. Devolver JSON para Laravel
# -----------------------------
print(json.dumps({
    "importados": imported,
    "atualizados": updated,
    "inalterados": unchanged
}))


{"importados": 0, "atualizados": 1, "inalterados": 14}
